# Lesson 02 - 探索 Microsoft Agent Framework

**Microsoft Agent Framework (MAF)** 是一個用於構建 AI 代理的統一框架。它提供了一個乾淨且可組合的架構，包含四個核心組件：

- **Client** – 連接到 AI 模型端點並處理通訊
- **Agent** – 將客戶端封裝為帶有指令和工具定義的代理
- **Tools** – 透過自訂功能擴展代理能力，讓模型可以調用
- **Session** – 維護多輪對話的會話歷史記錄

在本課中，我們將使用這些概念構建一個**旅遊預訂代理**，用以檢查目的地的可用性。


## 設置


In [ ]:
# Install the Microsoft Agent Framework package
! pip install agent-framework azure-ai-projects -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

## 了解 Agent Framework 架構

Microsoft Agent Framework 採用分層架構：

```
Client  →  Agent  →  Tools
                  →  Session
```

1. **Client** – 一個 `AzureAIProjectAgentProvider` 連接至 Azure OpenAI 部署。它負責認證、請求格式化及回應解析。
2. **Agent** – 從 client 透過 `provider.create_agent()` 建立，agent 結合模型存取、指令（系統提示）及工具。
3. **Tools** – 使用 `@tool` 裝飾的 Python 函數，agent 可調用它們來執行動作或取得資料。
4. **Session** – 一個 `AgentSession` 物件（透過 `agent.create_session()` 建立），儲存對話歷史，使多輪對話中 agent 能記住先前上下文。

讓我們一步步建立每個層級。


In [ ]:
# Create the client – this is the connection to the AI model
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## 使用 @tool 裝飾器新增工具

工具讓代理能執行超出產生文字之外的行動。`@tool` 裝飾器將一般的 Python 函式轉換成代理可以呼叫的工具。

重點：
- 使用 `Annotated[type, "description"]` 讓模型理解每個參數。
- 文件字串成為模型看到的工具描述。
- `approval_mode="never_require"` 表示工具會自動執行，無需用戶確認。


In [ ]:
@tool(approval_mode="never_require")
def check_destination_availability(
    destination: Annotated[str, "The destination to check availability for"]
) -> str:
    """Check if a vacation destination is currently available for booking."""
    available = {
        "Barcelona": True,
        "Tokyo": True,
        "Cape Town": False,
        "Vancouver": True,
        "Dubai": False,
    }
    is_available = available.get(destination, False)
    return f"{destination} is {'available' if is_available else 'not available'} for booking."

## 使用工具建立代理人

現在我們將客戶端、指令和工具結合成一個代理人。`instructions` 作為系統提示 —— 它們定義了代理人的個性和行為。


In [ ]:
agent = await provider.create_agent(
    name="TravelAvailabilityAgent",
    instructions=(
        "You are a travel booking agent. Help users check destination availability "
        "and make recommendations. Always check availability before recommending a destination."
    ),
    tools=[check_destination_availability],
)

## 多輪對話與會話管理

一個 `AgentSession`（通過 `agent.create_session()` 建立）會追蹤對話中的所有訊息。透過在每次呼叫 `agent.run()` 時傳入相同的會話，代理程式即可取得完整的對話歷史，並能回顧之前的訊息。

我們傳入 `tools=[check_destination_availability]`，讓代理程式能在每輪對話中調用我們的可用性檢查工具。


In [ ]:
session = agent.create_session()

# Turn 1: Ask about available destinations
response = await agent.run(
    "Which destinations do you have available?",
    session=session,
)
print(f"Agent: {response}")

In [ ]:
# Turn 2: Follow-up question — the agent remembers the conversation
response = await agent.run(
    "I'd like to go somewhere warm. What's available?",
    session=session,
)
print(f"Agent: {response}")

## Summary

在此課程中，您探索了 Microsoft Agent Framework 的四大支柱：

| Concept | What You Learned |
|---------|------------------|
| **Client** | `AzureAIProjectAgentProvider` 使用基於憑證的身份驗證連接至 Azure OpenAI |
| **Agent** | `provider.create_agent()` 將模型連接、指令及名稱打包在一起 |
| **Tools** | `@tool` 裝飾器可將 Python 函數暴露給代理呼叫 |
| **Session** | `agent.create_session()` 在多輪對話中維持對話歷史 |

這些組成部分共同創建了能進行自然對話、呼叫外部函數並維持語境的代理——為後續課程中更進階的代理模式奠定基礎。


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責聲明**：  
本文件經由人工智能翻譯服務 [Co-op Translator](https://github.com/Azure/co-op-translator) 進行翻譯。儘管我們力求準確，但請注意，自動翻譯可能包含錯誤或不準確之處。原始文件以其母語版本為唯一權威來源。對於重要資訊，建議採用專業人工翻譯。我們對因使用本翻譯而引起的任何誤解或誤釋概不負責。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
